## Desenvolvendo a solução MapReduce

### Map

In [51]:
number_list = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

exp_list = list(map(lambda x: x**2, number_list))
print(exp_list)

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


### Reduce

In [52]:
from functools import reduce

number_list = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
exp_list = list(map(lambda x: x**2, number_list))

exp_sum = reduce(lambda x, y: x+y, exp_list)
print(exp_sum)

285


## Demonstrando contagem de palavras

In [3]:
from urllib.request import Request, urlopen
import ssl

url = 'http://gutenberg.readingroo.ms/2/6/0/2600/2600.txt'

def request(url: str):
  ssl._create_default_https_context = ssl._create_unverified_context
  request = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
  response = urlopen(request)
  data = response.read()
  return data.decode('utf-8')


In [54]:
from urllib.request import Request, urlopen

url = 'http://gutenberg.readingroo.ms/2/6/0/2600/2600.txt'
response = request(url)
text = response[627:]

In [55]:
print(text[:37])

WAR AND PEACE

By Leo Tolstoy/Tolstoi


In [56]:
words = text.split()
print ('Number of words: %i' % len(words))

Number of words: 566218


In [57]:
import os

if os.name == "nt":
    # Safer multithreading on Windows
    from multiprocessing.dummy import Pool
else:
    #Multiprocessing on Linux,Mac
    from multiprocessing import Pool
    
from multiprocessing import cpu_count
from functools import partial

def remove_punctuation(text):
    return ''.join([letter for letter in text if letter not in ['.', ',', '!', '?', '"']])

def count_words(list_of_words, keywords):
    results = list()
    for word in list_of_words:
        for keyword in keywords:
            if keyword == remove_punctuation( word.upper()):
                results.append((keyword,1))
    return results

In [58]:
def partition(data, size):
    return [data[x:x+size] for x in range(0, len(data), size)]

def distribute(function, data, cores): 
    pool = Pool(cores)
    results = pool.map(function, data)
    pool.close()
    return results

def shuffle_sort(list):
    # Shuffle
    mapping = dict()
    for sublist in list:
        for key_pair in sublist:
            key, value = key_pair
            if key in mapping:
                mapping[key].append(key_pair)
            else:
                mapping[key] = [key_pair]
    return [mapping[key] for key in mapping]

def reduce(mapping):
  return (mapping[0][0], sum([value for (key, value) in mapping]))

In [59]:
cores = cpu_count()
print ('You have %i cores available for MapReduce' % cores)

You have 4 cores available for MapReduce


In [60]:
map_fn = partial(count_words, 
              keywords=['WAR', 'PEACE', 'RUSSIA', 
                        'NAPOLEON'])
map_result = distribute(map_fn, partition(words,len(words)//cores+1), cores)
print ('map_result is a list made of %i elements' % 
       len(map_result))
print ('Preview of one element: %s]'% map_result[0][:5])

map_result is a list made of 4 elements
Preview of one element: [('WAR', 1), ('PEACE', 1), ('WAR', 1), ('WAR', 1), ('RUSSIA', 1)]]


In [61]:
shuffled = shuffle_sort(map_result)
print ('Shuffled is a list made of %i elements' % 
       len(shuffled))
print ('Preview of first element: %s]'% shuffled[0][:5])
print ('Preview of second element: %s]'% shuffled[1][:5])

Shuffled is a list made of 4 elements
Preview of first element: [('WAR', 1), ('WAR', 1), ('WAR', 1), ('WAR', 1), ('WAR', 1)]]
Preview of second element: [('PEACE', 1), ('PEACE', 1), ('PEACE', 1), ('PEACE', 1), ('PEACE', 1)]]


In [62]:
result = distribute(reduce, shuffled, cores)
print ('Emitted results are: %s' % result)

Emitted results are: [('WAR', 288), ('PEACE', 111), ('RUSSIA', 156), ('NAPOLEON', 469)]


In [2]:
import urllib.request
url = "https://gutenberg.pglaf.org/1/6/6/1661/old/1661.txt"

response = request(url)
text = response[723:]
words = text.split()

URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Hostname mismatch, certificate is not valid for 'gutenberg.pglaf.org'. (_ssl.c:1129)>

In [64]:
print (text[:65])
print ('\nTotal words are %i' % len(words))

THE ADVENTURES OF SHERLOCK HOLMES

by

SIR ARTHUR CONAN DOYLE

Total words are 107431


In [65]:
map_fn = partial(count_words, keywords=['WATSON', 'ELEMENTARY'])
result = distribute(reduce, shuffle_sort(distribute(map_fn, partition(words,len(words)//cores), cores)), 1)
print ('Emitted results are: %s' % result)

Emitted results are: [('WATSON', 81), ('ELEMENTARY', 1)]
